# 03. More Prefix Patterns and 2D Prefix Sum

Same idea in different clothes: **left side and right side**, **prefix products**, **counts**, and **2D grids**.

**Learning order:** PIVOT INDEX → PRODUCT EXCEPT SELF → PREFIX OF COUNTS → 2D PREFIX SUM → INTERVIEW THINKING

## 1. Pivot Index (left sum = right sum)

For index `i`: `left = sum before i`, `right = total - left - nums[i]`. Keep a running `left` and compare. No extra array needed.

Real life: finding the balance point of a seesaw with weights on both sides.

In [1]:
def pivot_index(nums):
    total = sum(nums)
    left = 0
    for i, x in enumerate(nums):
        if left == total - left - x:
            return i
        left += x
    return -1

print(pivot_index([1, 7, 3, 6, 5, 6]))   # 3
print(pivot_index([1, 2, 3]))            # -1

3
-1


## 2. Product of Array Except Self

Answer at `i` = (product of everything **left** of `i`) × (product of everything **right** of `i`). No division needed (zeros break division).

Pass 1 left to right stores left products. Pass 2 right to left multiplies the right products with a running variable. O(n) time, O(1) extra space (output not counted).

In [2]:
def product_except_self(nums):
    n = len(nums)
    result = [1] * n
    left = 1
    for i in range(n):
        result[i] = left
        left *= nums[i]
    right = 1
    for i in range(n - 1, -1, -1):
        result[i] *= right
        right *= nums[i]
    return result

print(product_except_self([1, 2, 3, 4]))   # [24, 12, 8, 6]

[24, 12, 8, 6]


## 3. Prefix sums work on counts too

"How many vowels between positions l and r?" Make `prefix[i]` = number of vowels in the first `i` characters. Then it is the same formula. Real life: counting how many orders came in a time range from a running total.

In [3]:
def vowel_prefix(s):
    prefix = [0]
    for ch in s:
        prefix.append(prefix[-1] + (ch in "aeiou"))
    return prefix

p = vowel_prefix("interview")
print(p[4] - p[0])   # vowels in "inte" -> 2
print(p[9] - p[3])   # vowels in "erview" -> 3  (e, i, e)

2
3


## 4. 2D prefix sum (sum of any rectangle)

`p[i][j]` = sum of the rectangle from `(0, 0)` to `(i-1, j-1)`.

**Build (inclusion–exclusion):**
`p[i+1][j+1] = matrix[i][j] + p[i][j+1] + p[i+1][j] - p[i][j]`
(add the top and left rectangles, subtract the overlap counted twice)

**Query** rectangle `(r1,c1)` to `(r2,c2)`:
`p[r2+1][c2+1] - p[r1][c2+1] - p[r2+1][c1] + p[r1][c1]`
(big rectangle, remove the top strip and left strip, add back the corner removed twice)

In [4]:
class NumMatrix:
    def __init__(self, matrix):
        m, n = len(matrix), len(matrix[0])
        self.p = [[0] * (n + 1) for _ in range(m + 1)]
        for i in range(m):
            for j in range(n):
                self.p[i + 1][j + 1] = (matrix[i][j] + self.p[i][j + 1]
                                        + self.p[i + 1][j] - self.p[i][j])

    def sum_region(self, r1, c1, r2, c2):
        p = self.p
        return p[r2 + 1][c2 + 1] - p[r1][c2 + 1] - p[r2 + 1][c1] + p[r1][c1]

grid = [[3, 0, 1, 4, 2],
        [5, 6, 3, 2, 1],
        [1, 2, 0, 1, 5],
        [4, 1, 0, 1, 7],
        [1, 0, 3, 0, 5]]
nm = NumMatrix(grid)
print(nm.sum_region(2, 1, 4, 3), nm.sum_region(1, 1, 2, 2), nm.sum_region(1, 2, 2, 4))   # 8 11 12

8 11 12


## 5. Interview thinking

**Signals:** "sum on the left equals sum on the right", "everything except itself", "many rectangle sum queries", "count in a range".

**Say:** "I precompute a running total in one pass so each query becomes a subtraction."

**Common mistakes:** missing the extra row and column of zeros in 2D, wrong sign in inclusion–exclusion, using division in Product Except Self.

## Quick Revision

- Pivot: `left == total - left - nums[i]`.
- Product except self = left product × right product, two passes.
- Prefix works for counts too.
- 2D build: `a + top + left - corner`. Query: `big - top - left + corner`.